# Week 65, Day 6: Forge

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w065d6.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
import timeit
t = timeit.timeit("x = 1", number=10, timer=iter([3.0, 3.5]).__next__)
print(t, t / 10)

**Predict before you run:** what will this print?



In [ ]:
ops = 0
items = list(range(50))
for x in items:
    ops += 1
    if x in items[:x]:
        pass
print(ops)

**Predict before you run:** what will this print?



In [ ]:
import heapq, bisect
print(heapq.nsmallest(2, [5, 3, 8, 1]), bisect.bisect_right([1, 2, 2, 3], 2))

**Predict before you run:** what will this print?



In [ ]:
from functools import lru_cache
@lru_cache(maxsize=None)
def f(x):
    return x * 2
[f(i % 3) for i in range(10)]
print(f.cache_info().hits, f.cache_info().misses)

**Predict before you run:** what will this print?



In [ ]:
import numpy as np
m = np.array([[1, 2], [3, 4], [5, 6]])
print(m.sum(axis=0).tolist(), m.sum(axis=1).tolist(), (m > 2).sum())

### Exercise: Spotting nested loops

Using `ast` (Week 64), write `nested_loops(source)`: the sorted names of functions that contain a `for` or `while` loop **inside** another `for` or `while` loop, counting a comprehension inside a loop as a nested loop too. A single comprehension on its own doesn't count. Nested loops aren't always slow, but they're the first place to look for hidden O(n²) code.

In [ ]:
import ast

def nested_loops(source):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_nested():\n    \"finding possible O(n\u00b2) code\"\n    src = (\n        \"def pairs(xs):\\n    out = []\\n    for a in xs:\\n        for b in xs:\\n            out.append((a, b))\\n    return out\\n\"\n        \"def total(xs):\\n    t = 0\\n    for x in xs:\\n        t += x\\n    return t\\n\"\n        \"def hidden(xs):\\n    for x in xs:\\n        ys = [y for y in xs if y > x]\\n    return ys\\n\"\n        \"def waits(n):\\n    while n:\\n        n -= 1\\n        while False:\\n            pass\\n\"\n        \"def flat(xs):\\n    return [x for x in xs]\\n\"\n    )\n    assert nested_loops(src) == [\"hidden\", \"pairs\", \"waits\"]\n    assert nested_loops(\"x = 1\") == []")

### Exercise: Processing in chunks

Write a generator `chunked(iterable, size)` (Week 59) yielding lists of up to `size` items, so huge inputs can be processed a piece at a time without holding everything in memory, and a generator `chunk_sums(iterable, size)` yielding the sum of each chunk. Both must be lazy: asking for the first chunk takes only the first `size` items from the input.

In [ ]:
def chunked(iterable, size):
    ___

def chunk_sums(iterable, size):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_chunked():\n    \"streaming in chunks\"\n    import types, itertools\n    assert list(chunked(range(7), 3)) == [[0, 1, 2], [3, 4, 5], [6]] and list(chunked([], 2)) == []\n    gen = chunked(itertools.count(), 1000)\n    assert isinstance(gen, types.GeneratorType) and next(gen)[-1] == 999 and next(gen)[0] == 1000\n    assert list(chunk_sums(range(10), 4)) == [6, 22, 17]\n    pulled = []\n    def source_items():\n        for i in range(100):\n            pulled.append(i)\n            yield i\n    sums = chunk_sums(source_items(), 10)\n    next(sums)\n    assert len(pulled) == 10")

### Exercise: Forge: a faster log analyser

Write fast versions of all five functions, with exactly the same results (including order and tie-breaking): `unique_visitors` in first-seen order; `top_pages` as `(page, count)` pairs, most visited first, ties alphabetical; `visits_by_country` calling `lookup` **once per distinct address**; `busiest_window` returning `(count, start)` for the most events in any window `[a, a + width)` that starts at an event time (the earliest such start on ties; `(0, None)` for no events), using a sort and a single sweep; and `sessions`, counting each visitor's sessions (a new session starts after a gap **longer** than `gap`) with **one** pass over the log. The checks count equality tests, comparisons, lookup calls and passes over the log.

In [ ]:
import bisect
import heapq
from collections import Counter, defaultdict

def unique_visitors(log):
    ___

def top_pages(log, k):
    ___

def visits_by_country(log, lookup):
    ___

def busiest_window(times, width):
    ___

def sessions(log, gap):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def _log(n, seed):\n    import random\n    random.seed(seed)\n    ips = [f\"10.0.0.{i}\" for i in range(40)]\n    pages = [\"/\", \"/about\", \"/shop\", \"/cart\", \"/help\", \"/blog\"]\n    t, log = 0, []\n    for _ in range(n):\n        t += random.choice([1, 1, 2, 5, 30, 400])\n        log.append({\"time\": t, \"ip\": random.choice(ips), \"page\": random.choice(pages)})\n    return log\n\ndef test_visitors_and_pages():\n    \"unique visitors and top pages\"\n    class Ip(str):\n        eq_calls = 0\n        def __eq__(self, other):\n            Ip.eq_calls += 1\n            return str.__eq__(self, other)\n        __hash__ = str.__hash__\n    log = _log(3000, 1)\n    expected = []\n    for e in log:\n        if e[\"ip\"] not in expected:\n            expected.append(e[\"ip\"])\n    wrapped = [dict(e, ip=Ip(e[\"ip\"])) for e in log]\n    Ip.eq_calls = 0\n    assert unique_visitors(wrapped) == expected and Ip.eq_calls < 6000\n    counts = {}\n    for e in log:\n        counts[e[\"page\"]] = counts.get(e[\"page\"], 0) + 1\n    slow = sorted(counts.items(), key=lambda pc: (-pc[1], pc[0]))[:3]\n    assert top_pages(log, 3) == slow and top_pages([], 2) == []\n    assert top_pages([{\"page\": \"b\"}, {\"page\": \"a\"}], 5) == [(\"a\", 1), (\"b\", 1)]\n\ndef test_countries():\n    \"an expensive lookup called once per address\"\n    log = _log(2000, 2)\n    calls = []\n    def lookup(ip):\n        calls.append(ip)\n        return \"GB\" if int(ip.split(\".\")[-1]) % 3 else \"NG\"\n    result = visits_by_country(log, lookup)\n    assert len(calls) == len(set(e[\"ip\"] for e in log)) == len(set(calls))\n    expected = {}\n    for e in log:\n        c = lookup(e[\"ip\"])\n        expected[c] = expected.get(c, 0) + 1\n    assert result == dict(sorted(expected.items())) and list(result) == [\"GB\", \"NG\"]\n\ndef test_busiest_window():\n    \"the busiest window, without comparing every pair\"\n    class T(int):\n        comparisons = 0\n        def __lt__(self, o):\n            T.comparisons += 1\n            return int(self) < int(o)\n        def __le__(self, o):\n            T.comparisons += 1\n            return int(self) <= int(o)\n        def __gt__(self, o):\n            T.comparisons += 1\n            return int(self) > int(o)\n        def __ge__(self, o):\n            T.comparisons += 1\n            return int(self) >= int(o)\n        def __sub__(self, o):\n            return T(int(self) - int(o))\n    def slow(times, width):\n        best, start = 0, None\n        for a in sorted(times):\n            count = sum(1 for b in times if a <= b < a + width)\n            if count > best:\n                best, start = count, a\n        return best, start\n    assert busiest_window([1, 2, 3, 10, 11, 30], 5) == (3, 1)\n    assert busiest_window([5, 1, 9, 2], 2) == (2, 1) and busiest_window([], 3) == (0, None)\n    times = [e[\"time\"] for e in _log(600, 3)]\n    assert busiest_window(times, 60) == slow(times, 60)\n    T.comparisons = 0\n    big = [T(e[\"time\"]) for e in _log(3000, 4)][::-1]\n    busiest_window(big, 50)\n    assert T.comparisons < 120000\n\ndef test_sessions():\n    \"sessions per visitor in one pass\"\n    class Counted(list):\n        passes = 0\n        def __iter__(self):\n            Counted.passes += 1\n            return super().__iter__()\n    log = [{\"time\": t, \"ip\": ip, \"page\": \"/\"} for t, ip in\n           [(1, \"a\"), (5, \"b\"), (8, \"a\"), (100, \"a\"), (101, \"b\"), (300, \"a\"), (302, \"a\")]]\n    assert sessions(log, 30) == {\"a\": 3, \"b\": 2}\n    Counted.passes = 0\n    big = Counted(_log(2000, 5))\n    result = sessions(big, 120)\n    assert Counted.passes <= 1 and sum(result.values()) >= len(result)\n    assert sessions([], 10) == {}")

> **Where the speed came from.** None of today's fixes were clever tricks with Python's internals. Each replaced a repeated scan with a structure built once: a dict keeps order and finds duplicates, a Counter counts in one pass, a dict of lookup results stops repeated calls, sorting once makes a sweep possible, and grouping once replaces a scan per visitor. That's the usual shape of real performance work: the profile points at a function, and the fix is a better algorithm or data structure, checked by tests that prove the results haven't changed. Micro-tuning comes last, if at all.

### Exercise: Forge bonus: summaries with NumPy

Write these summaries with NumPy and **no Python `for` statements**:

- `hourly_counts(times)`: times are seconds since midnight on the first day (they can run into later days); return a list of 24 counts of events per hour of the day, using `// 3600 % 24` and `np.bincount` with `minlength=24`;
- `moving_average(values, window)`: the averages of every run of `window` consecutive values, rounded to 3 places, using cumulative sums (`np.cumsum`) so each average costs O(1); an empty list if there are fewer values than `window`;
- `outliers(values, z=2.0)`: the indices of values more than `z` standard deviations from the mean (none if the standard deviation is 0);
- `response_summary(ms)`: a dict with `"median"` and `"p95"` (from `np.percentile`, rounded to 1 place) and `"slow_share"` (the fraction of responses over 1000 ms, rounded to 3 places), all Python floats.

In [ ]:
import numpy as np

def hourly_counts(times):
    ___

def moving_average(values, window):
    ___

def outliers(values, z=2.0):
    ___

def response_summary(ms):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_hours_and_averages():\n    \"bincount and cumsum\"\n    times = [0, 59, 3600, 3600 * 5 + 10, 86400 + 3600 * 5, 3600 * 23]\n    counts = hourly_counts(times)\n    assert len(counts) == 24 and counts[0] == 2 and counts[1] == 1 and counts[5] == 2 and counts[23] == 1 and sum(counts) == 6\n    assert moving_average([1, 2, 3, 4, 5], 2) == [1.5, 2.5, 3.5, 4.5]\n    assert moving_average([1, 2, 3], 3) == [2.0] and moving_average([1], 3) == []\n    import random\n    random.seed(9)\n    vals = [random.random() for _ in range(300)]\n    slow = [round(sum(vals[i:i + 7]) / 7, 3) for i in range(len(vals) - 6)]\n    fast = moving_average(vals, 7)\n    assert len(fast) == len(slow) and all(abs(a - b) < 1e-9 for a, b in zip(fast, slow))\n    import re\n    assert not re.search(r\"^\\s*for \", source, re.M)\n\ndef test_outliers_and_summary():\n    \"masks and percentiles\"\n    data = [10, 11, 9, 10, 12, 10, 95, 11, 10, -60]\n    assert outliers(data, 1.5) == [6, 9] and outliers([5, 5, 5]) == []\n    s = response_summary([100, 200, 300, 400, 2000])\n    assert s == {\"median\": 300.0, \"p95\": 1680.0, \"slow_share\": 0.2}\n    assert all(type(v) is float for v in s.values())")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

## Warm-up

If any surprised you, revisit [profiling](w065d1.html), [algorithms vs constants](w065d2.html), [fast data structures](w065d3.html), [caching](w065d4.html) or [vectorisation with NumPy](w065d5.html).

## Mixed review

## Forge: a faster log analyser

A website's access log is a list of entries like `{"time": 1712, "ip": "10.0.0.7", "page": "/shop"}`, in time order. Here's the current analyser. It's correct, but each function is slow:

```
def unique_visitors(log):                 # an 'in' scan for every entry
    seen = []
    for e in log:
        if e["ip"] not in seen:
            seen.append(e["ip"])
    return seen

def top_pages(log, k):                    # counting by scanning, then sorting everything
    pages = [e["page"] for e in log]
    counts = [(p, pages.count(p)) for p in set(pages)]
    return sorted(counts, key=lambda pc: (-pc[1], pc[0]))[:k]

def visits_by_country(log, lookup):       # calls a slow lookup for every entry
    totals = {}
    for e in log:
        country = lookup(e["ip"])
        totals[country] = totals.get(country, 0) + 1
    return dict(sorted(totals.items()))

def busiest_window(times, width):         # compares every pair of times
    best, start = 0, None
    for a in sorted(times):
        count = sum(1 for b in times if a <= b < a + width)
        if count > best:
            best, start = count, a
    return best, start

def sessions(log, gap):                   # scans the whole log for every visitor
    result = {}
    for ip in unique_visitors(log):
        times = sorted(e["time"] for e in log if e["ip"] == ip)
        result[ip] = 1 + sum(1 for a, b in zip(times, times[1:]) if b - a > gap)
    return dict(sorted(result.items()))
```

## Forge bonus: summaries with NumPy

## Weekly quiz

## Week 65 review

Tick off each one honestly. If one feels shaky, go back and redo two exercises from that day.

- I can time code fairly, profile it with cProfile, read pstats and measure memory with tracemalloc. ([Day 1](w065d1.html))
- I can tell when the algorithm or the constants matter, find hidden O(n²) code and hoist repeated work. ([Day 2](w065d2.html))
- I can choose among lists, dicts, sets, deques, heaps, bisect, Counter and array for the operations I need. ([Day 3](w065d3.html))
- I can cache safely, bound caches by size or time, build keys and invalidate results. ([Day 4](w065d4.html))
- I can vectorise numerical code with NumPy and avoid its pitfalls. ([Day 5](w065d5.html))
- I made a slow log analyser fast and summarised data with NumPy. (Today)

Tomorrow is a **rest day**. Next week: **Concurrency I**: processes and threads, `threading`, the GIL, race conditions and locks, and `concurrent.futures`.